In [1]:
import numpy as np 
import pandas as pd 
import warnings
warnings.filterwarnings('ignore')

In [2]:
train = pd.read_csv(r'../input/tabular-playground-series-nov-2021/train.csv')
test = pd.read_csv(r'../input/tabular-playground-series-nov-2021/test.csv')
submission = pd.read_csv(r'../input/tabular-playground-series-nov-2021/sample_submission.csv')

In [3]:
train.shape, test.shape, submission.shape

((600000, 102), (540000, 101), (540000, 2))

In [4]:
train.head()

,id,f0,f1,f2,f3,f4,f5,f6,f7,f8,...,f91,f92,f93,f94,f95,f96,f97,f98,f99,target
0,0,0.106643,3.59437,132.8040,3.18428,0.081971,1.18859,3.73238,2.266270,2.09959,...,1.09862,0.013331,-0.011715,0.052759,0.065400,4.211250,1.97877,0.085974,0.240496,0
1,1,0.125021,1.67336,76.5336,3.37825,0.099400,5.09366,1.27562,-0.471318,4.54594,...,3.46017,0.017054,0.124863,0.154064,0.606848,-0.267928,2.57786,-0.020877,0.024719,0
2,2,0.036330,1.49747,233.5460,2.19435,0.026914,3.12694,5.05687,3.849460,1.80187,...,4.88300,0.085222,0.032396,0.116092,-0.001688,-0.520069,2.14112,0.124464,0.148209,0
3,3,-0.014077,0.24600,779.9670,1.89064,0.006948,1.53112,2.69800,4.517330,4.50332,...,3.47439,-0.017103,-0.008100,0.062013,0.041193,0.511657,1.96860,0.040017,0.044873,0
4,4,-0.003259,3.71542,156.1280,2.14772,0.018284,2.09859,4.15492,-0.038236,3.37145,...,1.91059,-0.042943,0.105616,0.125072,0.037509,1.043790,1.07481,-0.012819,0.072798,1


In [5]:
train.drop('id',axis=1,inplace=True)
test.drop('id',axis=1,inplace=True)

### **Data Exploration:**

In [6]:
print('train: ')
train.describe().T.style.bar(subset=['mean'], color='#606ff2')\
                            .background_gradient(subset=['std'], cmap='PuBu')\
                            .background_gradient(subset=['50%'], cmap='PuBu')

train: 


,count,mean,std,min,25%,50%,75%,max
f0,600000.000000,0.306508,0.522450,-3.797450,0.026222,0.097788,0.397184,8.781500
f1,600000.000000,2.497590,1.554018,-1.223960,1.186238,2.516500,3.787630,6.226720
f2,600000.000000,306.644536,551.743893,-1842.530000,43.573400,133.626000,302.262250,6119.280000
f3,600000.000000,2.647901,1.544529,-1.368560,1.442028,2.634130,3.907640,6.521150
f4,600000.000000,0.177850,0.417488,-3.206210,0.019709,0.061586,0.112712,8.265470
f5,600000.000000,2.556832,1.562527,-1.169770,1.261038,2.590425,3.813662,6.515070
f6,600000.000000,2.699650,1.564000,-1.059310,1.385820,2.801255,3.996913,6.586780
f7,600000.000000,2.571593,1.549361,-1.281970,1.333848,2.557985,3.823450,6.258770
f8,600000.000000,2.538273,1.532988,-1.242020,1.292163,2.475880,3.804360,6.389670
f9,600000.000000,0.134370,0.421892,-2.577840,0.019563,0.058752,0.101046,7.078460


In [7]:
print('test: ')
test.describe().T.style.bar(subset=['mean'], color='#606ff2')\
                            .background_gradient(subset=['std'], cmap='PuBu')\
                            .background_gradient(subset=['50%'], cmap='PuBu')

test: 


,count,mean,std,min,25%,50%,75%,max
f0,540000.000000,0.348663,0.566251,-3.628650,0.043555,0.115868,0.457940,8.666950
f1,540000.000000,2.618251,1.543507,-1.260150,1.326280,2.657140,3.884850,6.434070
f2,540000.000000,263.577730,496.444309,-1764.160000,27.361175,115.631000,245.446250,6098.190000
f3,540000.000000,2.583735,1.529646,-1.370560,1.408300,2.547380,3.812990,6.275570
f4,540000.000000,0.211793,0.468154,-3.868460,0.028489,0.072008,0.125637,7.915200
f5,540000.000000,2.709543,1.568074,-1.180650,1.400490,2.797510,3.982830,6.444200
f6,540000.000000,2.762370,1.586975,-1.126370,1.433133,2.892945,4.083213,6.726750
f7,540000.000000,2.479384,1.520741,-1.284250,1.278905,2.456350,3.694180,6.339400
f8,540000.000000,2.549483,1.520833,-1.084660,1.306570,2.485895,3.811365,6.180310
f9,540000.000000,0.125970,0.399739,-2.692380,0.019464,0.057811,0.098911,7.819170


### **Data Preprocessing:**

In [8]:
num_features = ['f0', 'f1', 'f2', 'f3', 'f4', 'f5', 'f6', 'f7', 'f8', 'f9', 'f10', 
                'f11', 'f12', 'f13', 'f14', 'f15', 'f16', 'f17', 'f18', 'f19', 'f20', 
                'f21', 'f23', 'f24', 'f25', 'f26', 'f27', 'f28', 'f29', 'f30', 
                'f31', 'f32', 'f33', 'f34', 'f35', 'f36', 'f37', 'f38', 'f39', 'f40', 
                'f41', 'f42', 'f44', 'f45', 'f46', 'f47', 'f48', 'f49', 'f50', 
                'f51', 'f52', 'f53', 'f54', 'f55', 'f56', 'f57', 'f58', 'f59', 'f60', 
                'f61', 'f62', 'f63', 'f64', 'f65', 'f66', 'f67', 'f68', 'f69', 'f70', 
                'f71', 'f72', 'f73', 'f74', 'f75', 'f76', 'f77', 'f78', 'f79', 'f80', 
                'f81', 'f82', 'f83', 'f84', 'f85', 'f86', 'f87', 'f88', 'f89', 'f90', 
                'f91', 'f92', 'f93', 'f94', 'f95', 'f96', 'f97', 'f98', 'f99']

In [9]:
from sklearn.preprocessing import StandardScaler
ss = StandardScaler()
train[num_features] = ss.fit_transform(train[num_features])
test[num_features] = ss.transform(test[num_features])

### **PyCaret**

In [10]:
! pip install pycaret

     |████████████████████████████████| 266 kB 284 kB/s 
     |████████████████████████████████| 24.2 MB 1.8 MB/s 
     |████████████████████████████████| 25.9 MB 800 kB/s 
     |████████████████████████████████| 16.9 MB 57.0 MB/s 
     |████████████████████████████████| 113 kB 50.9 MB/s 
     |████████████████████████████████| 167 kB 46.1 MB/s 
     |████████████████████████████████| 1.1 MB 51.2 MB/s 
     |████████████████████████████████| 79 kB 7.3 MB/s 
     |████████████████████████████████| 58 kB 5.2 MB/s 
     |████████████████████████████████| 1.7 MB 50.1 MB/s 
  Installing build dependencies ... - \ | / done
  Getting requirements to build wheel ... - done
  Installing backend dependencies ... - \ | done
    Preparing wheel metadata ... - done
     |████████████████████████████████| 1.7 MB 49.4 MB/s 
  Created wheel for alembic: filename=alembic-1.4.1-py2.py3-none-any.whl size=158155 sha256=f873e16b15690ca41547eb71901e5eb6254a209c0e5617c03b9b54ddd3622942
  St

In [11]:
from pycaret.classification import setup, compare_models, blend_models, get_config

In [12]:
def pycaret_model(train, target,test, n_select, fold,opt):
    print('Setup Your Data....')
    setup(data=train,
          target=target,
          silent= True,use_gpu = True)
  
    print('Comparing Models....')
    best = compare_models(sort = opt,n_select=n_select, fold = fold,include = ['lightgbm','xgboost','catboost'])
    
    print('Blending Models....')
    blended = blend_models(estimator_list= best, fold=fold, optimize=opt)
    
    prep_pipe = get_config("prep_pipe")
    prep_pipe.steps.append(['final_model', blended])
    predictions = prep_pipe.predict_proba(test)[:,1]

    return predictions


In [13]:
submission['target'] = pycaret_model(train,'target',test, 2, 5,'AUC')

,Accuracy,AUC,Recall,Prec.,F1,Kappa,MCC
0,0.7065,0.7401,0.7163,0.7065,0.7114,0.4128,0.4128
1,0.7073,0.7414,0.7204,0.7060,0.7131,0.4144,0.4145
2,0.7048,0.7372,0.7172,0.7039,0.7105,0.4095,0.4096
3,0.7055,0.7375,0.7200,0.7037,0.7117,0.4107,0.4108
4,0.7080,0.7408,0.7127,0.7103,0.7115,0.4160,0.4160
Mean,0.7064,0.7394,0.7173,0.7061,0.7116,0.4127,0.4127
SD,0.0012,0.0017,0.0028,0.0024,0.0008,0.0024,0.0023


In [14]:
submission.to_csv('submission.csv',index=False)

<div class="alert alert-info">
<h4>If you like this notebook, please upvote it! 
     Thank you! :)</h4>
</div>